# 说明  
本 notebook 配套教程第五章和第六章，将结合U-Net模型来探索PyTorch的模型定义方式和进阶训练技巧。  
下方每个“Point”对应于教程中每一节的内容。

In [ ]:
import os
import numpy as np
import collections
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

**要点 1：模型定义方式**  
PyTorch中自定义模型主要通过以下三种方式：  
- Sequential（顺序容器）  
- ModuleList（模块列表）  
- ModuleDict（模块字典）

In [ ]:
## 讲解点：使用ordered dict更有助于一目了然模型结构，对于之后模型修改也非常有帮助

In [ ]:
## Sequential：直接列表写法
import torch.nn as nn
net1 = nn.Sequential(
        nn.Linear(784, 256),
        nn.ReLU(),
        nn.Linear(256, 10), 
        )
print(net1)

In [ ]:
## Sequential：有序字典写法
import collections
import torch.nn as nn
net2 = nn.Sequential(collections.OrderedDict([
          ('fc1', nn.Linear(784, 256)),
          ('relu1', nn.ReLU()),
          ('fc2', nn.Linear(256, 10))
          ]))
print(net2)

In [ ]:
# 试一下
a = torch.rand(4,784)
out1 = net1(a)
out2 = net2(a)
print(out1.shape==out2.shape, out1.shape)

In [ ]:
## ModuleList：模块列表容器
net3 = nn.ModuleList([nn.Linear(784, 256), nn.ReLU()])
net3.append(nn.Linear(256, 10)) # # 类似List的append操作
print(net3[-1])  # 类似List的索引访问
print(net3)

In [ ]:
# 注意：ModuleList 只是保存模块的容器，并没有定义前向传播逻辑。
# 直接调用会触发 NotImplementedError，这里捕获错误，保留教学效果，也让 notebook 可以继续运行。
try:
    out3 = net3(a)
except NotImplementedError as err:
    print(f"ModuleList 不能直接作为网络调用：{err.__class__.__name__}")

In [ ]:
class Net3(nn.Module):
    def __init__(self):
        super().__init__()
        self.modulelist = nn.ModuleList([nn.Linear(784, 256), nn.ReLU()])
        self.modulelist.append(nn.Linear(256, 10))
    
    def forward(self, x):
        for layer in self.modulelist:
            x = layer(x)
        return x
net3_ = Net3()
out3_ = net3_(a)
print(out3_.shape)

In [ ]:
## ModuleDict：模块字典容器
net4 = nn.ModuleDict({
    'linear': nn.Linear(784, 256),
    'act': nn.ReLU(),
})
net4['output'] = nn.Linear(256, 10) # 添加
print(net4['linear']) # 访问
print(net4.output)

In [ ]:
# ModuleDict 同样只是保存模块的容器，并没有定义前向传播逻辑。
# 正确用法是把它放进 nn.Module，并在 forward 中明确调用各层。
try:
    out4 = net4(a)
except NotImplementedError as err:
    print(f"ModuleDict 不能直接作为网络调用：{err.__class__.__name__}")

**要点 2：利用模型块快速搭建复杂网络**  
下面我们开始探索如何利用模型块，快速构建U-Net网络
![img](./unet.png)  
组成U-Net的模型块主要有如下几个部分：  
1）每个子块内部的两次卷积（Double Convolution，双卷积）  
2）左侧模型块之间的下采样连接，即最大池化（Max pooling，下采样）  
3）右侧模型块之间的上采样连接（Up sampling）  
4）输出层的处理  

除模型块外，还有模型块之间的横向连接，输入和U-Net底部的连接等计算，这些单独的操作可以通过forward函数来实现。  
（参考：https://github.com/milesial/Pytorch-UNet ）

In [ ]:
import os
import numpy as np
import collections
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

In [ ]:
class DoubleConv(nn.Module):
    """两次卷积：卷积 => 可选 BN => ReLU"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


In [ ]:
class Down(nn.Module):
    """先用最大池化下采样，再做双卷积"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


In [ ]:
class Up(nn.Module):
    """先上采样，再做双卷积"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # 如果使用双线性插值，就用普通卷积来减少通道数
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # 输入格式是 CHW，即通道数、高度、宽度
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # 如果遇到 padding 尺寸不匹配问题，可以参考下面两个修复链接
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

In [ ]:
class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

In [ ]:
## 组装
class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

In [ ]:
unet = UNet(3,1)
unet

**要点 3：模型修改**  
这里我们假设最后的分割是多类别的（即mask不止0和1，还有2，3，4等值代表其他目标），需要对模型特定层进行修改。   
此外还有两种情况的模型修改方式，这里也做演示：  
- 添加额外输入  
- 添加额外输出

In [ ]:
## 修改特定层
import copy
unet1 = copy.deepcopy(unet)
unet1.outc

In [ ]:
b = torch.rand(1,3,224,224)
out_unet1 = unet1(b)
print(out_unet1.shape)

In [ ]:
unet1.outc = OutConv(64, 5)
unet1.outc

In [ ]:
out_unet1 = unet1(b)
print(out_unet1.shape)

In [ ]:
## 添加额外输入
class UNet2(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNet2, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x, add_variable):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        x = x + add_variable   #修改点
        logits = self.outc(x)
        return logits
unet2 = UNet2(3,1)

c = torch.rand(1,1,224,224)
out_unet2 = unet2(b, c)
print(out_unet2.shape)

In [ ]:
## 添加额外输出
class UNet3(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNet3, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits, x5  # 修改点
unet3 = UNet3(3,1)

c = torch.rand(1,1,224,224)
out_unet3, mid_out = unet3(b)
print(out_unet3.shape, mid_out.shape)

**要点 4：模型保存与读取**  
这里相应考虑单卡和多卡情况下的模型存取情况

In [ ]:
## 讲解点：回到jupyter的文件目录下，看保存的结果

In [ ]:
unet

In [ ]:
unet.state_dict()

In [ ]:
## CPU 或单卡：保存并读取整个模型
# map_location="cpu" 可以让没有 GPU 的机器也能读取模型。
def load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

torch.save(unet, "./unet_example.pth")
loaded_unet = load_checkpoint("./unet_example.pth")
loaded_unet.state_dict()

In [ ]:
## CPU 或单卡：保存并读取模型权重
torch.save(unet.state_dict(), "./unet_weight_example.pth")
loaded_unet_weights = load_checkpoint("./unet_weight_example.pth")
unet.load_state_dict(loaded_unet_weights)
unet.state_dict()

In [ ]:
## 多卡：保存并读取整个模型。注意模型层名称前多了 module。
## 这里不强制要求真实多卡环境；在 CPU 上包装 DataParallel 也能展示 state_dict 的命名差异。
## 不建议长期保存整个 DataParallel 模型，因为读取环境变化时更容易出兼容问题。
unet_mul = nn.DataParallel(copy.deepcopy(unet))
unet_mul

In [ ]:
torch.save(unet_mul, "./unet_mul_example.pth")
loaded_unet_mul = load_checkpoint("./unet_mul_example.pth")
loaded_unet_mul

In [ ]:
## 多卡：保存并读取模型权重。
torch.save(unet_mul.state_dict(), "./unet_weight_mul_example.pth")
loaded_unet_weights_mul = load_checkpoint("./unet_weight_mul_example.pth")
unet_mul_reloaded = nn.DataParallel(copy.deepcopy(unet))
unet_mul_reloaded.load_state_dict(loaded_unet_weights_mul)
unet_mul_reloaded.state_dict()

In [ ]:
# 如果保存的是整个模型，也建议提取内部模型或权重，再重新构建干净的新模型。
loaded_base_unet = loaded_unet_mul.module if isinstance(loaded_unet_mul, nn.DataParallel) else loaded_unet_mul
unet_mul_from_whole = nn.DataParallel(copy.deepcopy(loaded_base_unet))
unet_mul_from_whole.state_dict()

**接下来进入第六章的内容，我们以前面已经搭建好的U-Net模型为例，探索如何更优雅地训练PyTorch模型。**  
**首先我们使用 [Carvana](https://www.kaggle.com/competitions/carvana-image-masking-challenge) 数据集，实现一个基本的 U-Net 训练过程。**

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from torchvision import transforms
import torch.optim as optim
import matplotlib.pyplot as plt
import PIL
from sklearn.model_selection import train_test_split

# 训练演示默认优先使用 CUDA；没有 CUDA 时使用 CPU，避免在不同机器上被 .cuda() 卡住。
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前训练设备：{device}")

In [ ]:
class CarvanaDataset(Dataset):
    def __init__(self, base_dir, idx_list, mode="train", transform=None):
        self.base_dir = base_dir
        self.idx_list = list(idx_list)
        self.images = sorted(os.listdir(os.path.join(base_dir, "train")))
        self.mode = mode
        self.transform = transform
    
    def __len__(self):
        return len(self.idx_list)

    def __getitem__(self, index):
        image_file = self.images[self.idx_list[index]]
        mask_file = image_file[:-4] + "_mask.gif"
        image = PIL.Image.open(os.path.join(self.base_dir, "train", image_file)).convert("RGB")
        if self.mode == "train":
            mask = PIL.Image.open(os.path.join(self.base_dir, "train_masks", mask_file)).convert("L")
            if self.transform is not None:
                image = self.transform(image)
                mask = self.transform(mask)
                mask[mask != 0] = 1.0
            return image, mask.float()
        if self.transform is not None:
            image = self.transform(image)
        return image


def build_demo_loaders(batch_size=2, image_size=64):
    """构造一个很小的合成分割数据集，方便没有 Carvana 数据时也能完整跑通示例。"""
    generator = torch.Generator().manual_seed(42)
    images = torch.rand(4, 3, image_size, image_size, generator=generator)
    masks = (images[:, :1] > 0.5).float()
    dataset = TensorDataset(images, masks)
    train_data, val_data = random_split(dataset, [3, 1], generator=generator)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader


def prepare_carvana_or_demo_loaders(base_dir="./", image_size=64, batch_size=2):
    train_dir = os.path.join(base_dir, "train")
    mask_dir = os.path.join(base_dir, "train_masks")
    transform = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor()])

    if os.path.isdir(train_dir) and os.path.isdir(mask_dir) and os.listdir(mask_dir):
        idxs = list(range(len(os.listdir(mask_dir))))
        if len(idxs) > 1:
            train_idxs, val_idxs = train_test_split(idxs, test_size=0.3, random_state=42)
        else:
            train_idxs, val_idxs = idxs, idxs
        train_data = CarvanaDataset(base_dir, train_idxs, transform=transform)
        val_data = CarvanaDataset(base_dir, val_idxs, transform=transform)
        train_loader = DataLoader(train_data, batch_size=batch_size, num_workers=0, shuffle=True)
        val_loader = DataLoader(val_data, batch_size=batch_size, num_workers=0, shuffle=False)
        print(f"已读取本地 Carvana 数据：训练 {len(train_data)} 张，验证 {len(val_data)} 张。")
        return train_loader, val_loader

    print("未找到 ./train 和 ./train_masks，改用小型合成数据跑通训练流程。")
    return build_demo_loaders(batch_size=batch_size, image_size=image_size)


train_loader, val_loader = prepare_carvana_or_demo_loaders()

In [ ]:
image, mask = next(iter(train_loader))
plt.subplot(121)
plt.imshow(image[0,0])
plt.subplot(122)
plt.imshow(mask[0,0], cmap="gray")

In [ ]:
# 使用 Binary Cross Entropy Loss，之后我们会尝试替换为自定义的 loss。
unet = unet.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(unet.parameters(), lr=1e-3, weight_decay=1e-8)

In [ ]:
def dice_coeff(pred, target):
    eps = 0.0001
    num = pred.size(0)
    m1 = pred.view(num, -1)  # 展平成二维，便于计算重叠面积
    m2 = target.view(num, -1)  # 展平成二维，便于计算重叠面积
    intersection = (m1 * m2).sum()
    return (2. * intersection + eps) / (m1.sum() + m2.sum() + eps)


def train(epoch):
    unet.train()
    train_loss = 0
    for data, mask in train_loader:
        data, mask = data.to(device), mask.to(device)
        optimizer.zero_grad()
        output = unet(data)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * data.size(0)
    train_loss = train_loss / len(train_loader.dataset)
    print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch, train_loss))


def val(epoch):  
    print("current learning rate: ", optimizer.state_dict()["param_groups"][0]["lr"])
    unet.eval()
    val_loss = 0
    dice_score = 0
    with torch.no_grad():
        for data, mask in val_loader:
            data, mask = data.to(device), mask.to(device)
            output = unet(data)
            loss = criterion(output, mask)
            val_loss += loss.item() * data.size(0)
            dice_score += dice_coeff(torch.sigmoid(output).cpu(), mask.cpu()) * data.size(0)
    val_loss = val_loss / len(val_loader.dataset)
    dice_score = dice_score / len(val_loader.dataset)
    print('Epoch: {} \tValidation Loss: {:.6f}, Dice score: {:.6f}'.format(epoch, val_loss, dice_score))

In [ ]:
epochs = 1
for epoch in range(1, epochs + 1):
    train(epoch)
    val(epoch)

In [ ]:
import subprocess

if torch.cuda.is_available():
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    print(result.stdout or result.stderr)
else:
    print("当前环境没有可用 CUDA，跳过 nvidia-smi。")

**要点 5：自定义损失函数**  
如果我们不想使用交叉熵函数，而是想针对分割模型常用的Dice系数设计专门的loss，即DiceLoss，这时就需要我们自定义PyTorch的损失函数

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceLoss, self).__init__()
        
    def forward(self,inputs,targets,smooth=1):
        inputs = torch.sigmoid(inputs)       
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()                   
        dice = (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)  
        return 1 - dice


In [ ]:
newcriterion = DiceLoss()

unet.eval()
image, mask = next(iter(val_loader))
image, mask = image.to(device), mask.to(device)
out_unet = unet(image)
loss = newcriterion(out_unet, mask)
print(loss)

**要点 6：动态调整学习率**  
随着优化的进行，固定的学习率可能无法满足优化的需求，这时需要调整学习率，降低优化的速度  
这里演示使用PyTorch自带的StepLR scheduler动态调整学习率的效果，文字版教程中给出了自定义scheduler的方式

In [ ]:
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.8)

In [ ]:
epochs = 1
for epoch in range(1, epochs + 1):
    train(epoch)
    val(epoch)
    scheduler.step()

In [ ]:
print(optim.lr_scheduler.StepLR.__doc__.splitlines()[0])

**要点 7：模型微调**

In [ ]:
unet

In [ ]:
target_unet = unet.module if isinstance(unet, nn.DataParallel) else unet
target_unet.outc.conv.weight.requires_grad = False
target_unet.outc.conv.bias.requires_grad = False

for layer, param in unet.named_parameters():
    print(layer, '\t', param.requires_grad)

In [ ]:
param

**要点 8：半精度训练**

In [ ]:
## 演示混合精度：有 CUDA 时启用 autocast；没有 CUDA 时自动退化为普通精度，方便本地 CPU 环境运行。
from torch.cuda.amp import autocast

amp_enabled = torch.cuda.is_available()
print(f"是否启用 CUDA autocast：{amp_enabled}")

In [ ]:
# 复用上面定义的数据准备函数；没有 Carvana 数据时仍然使用小型合成数据。
train_loader, val_loader = prepare_carvana_or_demo_loaders()

In [ ]:
class UNet_half(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNet_half, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)
    
    @autocast(enabled=torch.cuda.is_available())
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits


unet_half = UNet_half(3, 1).to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(unet_half.parameters(), lr=1e-3, weight_decay=1e-8)

In [ ]:
def dice_coeff(pred, target):
    eps = 0.0001
    num = pred.size(0)
    m1 = pred.view(num, -1)  # 展平成二维，便于计算重叠面积
    m2 = target.view(num, -1)  # 展平成二维，便于计算重叠面积
    intersection = (m1 * m2).sum()
    return (2. * intersection + eps) / (m1.sum() + m2.sum() + eps)


def train_half(epoch):
    unet_half.train()
    train_loss = 0
    for data, mask in train_loader:
        data, mask = data.to(device), mask.to(device)
        optimizer.zero_grad()
        with autocast(enabled=amp_enabled):
            output = unet_half(data)
            loss = criterion(output, mask)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * data.size(0)
    train_loss = train_loss / len(train_loader.dataset)
    print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch, train_loss))


def val_half(epoch):  
    print("current learning rate: ", optimizer.state_dict()["param_groups"][0]["lr"])
    unet_half.eval()
    val_loss = 0
    dice_score = 0
    with torch.no_grad():
        for data, mask in val_loader:
            data, mask = data.to(device), mask.to(device)
            with autocast(enabled=amp_enabled):
                output = unet_half(data)
                loss = criterion(output, mask)
            val_loss += loss.item() * data.size(0)
            dice_score += dice_coeff(torch.sigmoid(output).cpu(), mask.cpu()) * data.size(0)
    val_loss = val_loss / len(val_loader.dataset)
    dice_score = dice_score / len(val_loader.dataset)
    print('Epoch: {} \tValidation Loss: {:.6f}, Dice score: {:.6f}'.format(epoch, val_loss, dice_score))

In [ ]:
epochs = 1
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.8)
for epoch in range(1, epochs + 1):
    train_half(epoch)
    val_half(epoch)
    scheduler.step()

In [ ]:
import subprocess

if torch.cuda.is_available():
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    print(result.stdout or result.stderr)
else:
    print("当前环境没有可用 CUDA，跳过 nvidia-smi。")